# EDA — QC, coordinate + scalefactor sanity, clustering

Orchestrates the `visium_spatial` package. The **smoke test** below runs the committed synthetic fixture (no download) through the graph + ranking to confirm the environment and the owned code work on a fresh clone. The **real run** loads the lymph node section via `load_visium` (see README *Data* / *How to run*).

## Smoke test — offline synthetic fixture

Deterministic, no download. Exercises the same code paths the test suite validates.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pathlib, sys
import numpy as np

# Offline smoke data: the project's committed synthetic fixture (no download).
_root = pathlib.Path.cwd()
_root = _root if (_root / "pyproject.toml").exists() else _root.parent
sys.path.insert(0, str(_root / "tests" / "fixtures"))
from synthetic import make_synthetic_visium

from visium_spatial.build_graph import build_spatial_graph, assert_hex_neighbors
from visium_spatial.global_autocorr import rank_svgs

adata = make_synthetic_visium(seed=0)
build_spatial_graph(adata)
print("hex check:", assert_hex_neighbors(adata))   # modal degree 6, isolates reported
ranked = rank_svgs(adata, seed=0)
ranked

## Real run — human lymph node section

Set `DATA_DIR` to your Space Ranger `outs/`. The cell is a no-op until the data is present, so the notebook still runs clean on a fresh clone.

In [ ]:
import pathlib
from visium_spatial.load_visium import load_visium, extract_scalefactors, assert_spatial_frame
from visium_spatial.qc import compute_qc_metrics, filter_spots, filter_genes
from visium_spatial.preprocess import normalize, select_hvg
from visium_spatial.cluster import leiden_clusters

DATA_DIR = pathlib.Path("data/lymph_node/outs")  # <-- edit to your section

if DATA_DIR.exists():
    adata = load_visium(DATA_DIR)
    assert_spatial_frame(adata)
    print("scalefactors:", extract_scalefactors(adata))
    compute_qc_metrics(adata)
    adata = filter_spots(adata, min_counts=500, min_genes=250, max_pct_mito=30.0)
    filter_genes(adata, min_spots=10)
    normalize(adata)
    select_hvg(adata, n_top_genes=2000)
    leiden_clusters(adata)
    print(adata)
else:
    print(f"No data at {DATA_DIR} — see README. The smoke test above still validates the code.")